## 1. 环境准备

In [ ]:
import sys; sys.path.insert(0, '../src')
import os
import json
import pandas as pd
from pathlib import Path

# 检查是否有 API Key
api_key = os.getenv('PRO_DATASET_API_KEY')
if api_key:
    mode = 'online'
    print(f'✅ 在线模式：检测到 PRO_DATASET_API_KEY（{api_key[:8]}...）')
else:
    mode = 'offline'
    print('⚠️  离线模式：未检测到 PRO_DATASET_API_KEY，将使用缓存数据')
    print('   设置环境变量 PRO_DATASET_API_KEY 可切换到在线模式')

CACHE_DIR = Path('../data/raw/cached_responses')

## 2. 数据检索封装（在线 + 离线）

In [ ]:
def search_data(query: str, cache_file: str = None):
    """
    统一数据检索接口。
    - 在线模式：调用 ProDatasetClient
    - 离线模式：读取缓存 JSON
    """
    if mode == 'online':
        from utils.pro_dataset_client import ProDatasetClient
        client = ProDatasetClient(api_key=api_key)
        result = client.search(query)
        if result.success:
            print(f'✅ 检索成功: dataset_type={result.dataset_type}')
            return result.raw
        else:
            print(f'❌ 检索失败: {result.msg}')
            if cache_file:
                print(f'   回退到缓存数据: {cache_file}')
    
    # 离线模式或在线失败回退
    if cache_file:
        cache_path = CACHE_DIR / cache_file
        if cache_path.exists():
            with open(cache_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            print(f'📦 使用缓存数据: {cache_file}')
            return data
    raise FileNotFoundError(f'缓存文件不存在: {cache_file}')

## 3. 检索比亚迪月度销量趋势

In [ ]:
# 在线调用 dataPro_search 等价查询
result = search_data(
    query='比亚迪 近一年全国销量趋势',
    cache_file='byd_monthly_trend.json'
)
print(json.dumps(result, ensure_ascii=False, indent=2)[:600])

In [ ]:
# 解析为 DataFrame
sales_data = result['items'][0]['销售数据']
byd_trend = pd.DataFrame(sales_data)
byd_trend['month'] = pd.to_datetime(byd_trend['时间'])
byd_trend['market_share'] = byd_trend['在售厂商份额'].str.rstrip('%').astype(float) / 100
byd_trend = byd_trend[['month', '销量', 'market_share', '在售厂商排名']]
byd_trend.columns = ['month', 'sales', 'market_share', 'rank']
display(byd_trend)

In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(byd_trend['month'].dt.strftime('%Y-%m'), byd_trend['sales'], alpha=0.7, label='销量')
ax1.set_ylabel('销量')
ax2 = ax1.twinx()
ax2.plot(byd_trend['month'].dt.strftime('%Y-%m'), byd_trend['market_share'], 'r-o', label='市场份额')
ax2.set_ylabel('市场份额')
plt.title('比亚迪月度销量与市场份额（真实数据）')
plt.xticks(rotation=45)
plt.tight_layout()

## 4. 检索比亚迪 2026年8月分车系销量

In [ ]:
result2 = search_data(
    query='比亚迪 2026年8月 各车系销量',
    cache_file='byd_aug_2026_models.json'
)

# 解析销售网络数据
networks = result2['items'][0]['销售数据'][0]['销售网络']
rows = []
for net in networks:
    for car in net.get('车系', []):
        rows.append({'network': net['网络名称'], 'model': car['车系名称'], 'sales': car['本月销量']})
byd_models = pd.DataFrame(rows)
display(byd_models)

In [ ]:
# 各网络销量占比
byd_models.groupby('network')['sales'].sum().plot(
    kind='pie', autopct='%1.1f%%', figsize=(8, 8), title='比亚迪 2026年8月 各网络销量占比'
)

## 5. 检索特斯拉数据并对比

In [ ]:
result3 = search_data(
    query='特斯拉 2026年8月全国销量',
    cache_file='tesla_aug_2026.json'
)
tesla_data = result3['items'][0]['销售数据']
tesla_df = pd.DataFrame(tesla_data)
display(tesla_df)

## 6. 宏观数据：新能源汽车销量

In [ ]:
result4 = search_data(
    query='2026年8月中国新能源汽车销量',
    cache_file='macro_nev_aug_2026.json'
)
macro_data = result4['items'][0]
print(f"2026年8月中国新能源汽车销量: {macro_data['value']:,.0f} {macro_data['unit_raw']}")

## 7. 端到端分析：比亚迪 vs 市场

In [ ]:
# 比亚迪 8月零售量 vs 全国新能源汽车销量
byd_aug_retail = byd_trend[byd_trend['month'] == '2026-08-01']['sales'].values[0]
nev_aug_total = macro_data['value']
byd_share = byd_aug_retail / nev_aug_total * 100

print(f'2026年8月:')
print(f'  比亚迪零售量: {byd_aug_retail:,} 辆')
print(f'  全国新能源汽车销量: {nev_aug_total:,.0f} 辆')
print(f'  比亚迪市占率: {byd_share:.1f}%')
print(f'  专业数据集返回市场份额: {byd_trend.iloc[-1]["market_share"]*100:.1f}%')

## 8. 在线模式进阶：自由查询

以下代码仅在**在线模式**下运行，可自由查询专业数据集的各类数据。

In [ ]:
if mode == 'online':
    from utils.pro_dataset_client import ProDatasetClient
    client = ProDatasetClient(api_key=api_key)
    
    # 示例：查询学术文献
    r = client.search('时序异常检测 近2年论文')
    print(f'文献检索: {r.msg}, 共 {len(r.items)} 条')
    if r.items:
        print(json.dumps(r.items[0], ensure_ascii=False, indent=2)[:400])
else:
    print('当前为离线模式，跳过自由查询演示。')
    print('设置 PRO_DATASET_API_KEY 环境变量后重新运行即可体验在线检索。')

## 小结

本 Notebook 演示了如何通过火山引擎「专业数据集」获取真实业务数据：

1. **在线模式**：通过 `ProDatasetClient` 直接调用 `dataPro_search` 工具
2. **离线模式**：使用项目内置的缓存 JSON 数据
3. **统一接口**：`search_data()` 自动切换两种模式
4. **真实分析**：基于真实数据完成销量趋势、车系结构、市场份额分析

切换到在线模式只需设置环境变量：
```bash
export PRO_DATASET_API_KEY=hqd_sk_your_key_here
```